# Flood-mask data quality EDA
Where do missing values, ranges and cardinality stand in the satellite flood mask data?
We profile the same Parquet files used by `EDA_flood_masks/flood_masks_eda.ipynb` at two levels:
- **Country (South Sudan)**: every record across both MODIS/VIIRS tiles (`h20v08` and `h21v08`)
  for both flood classes (`recurring` and `unusual`), years 2000-2025.
- **Northern Bahr el Ghazal state**: only records whose pixel centre falls inside the
  Northern Bahr el Ghazal admin-1 polygon (the five Aweil counties).

Because the whole-country dataset has ~95 million rows, we aggregate column statistics with
PyArrow (one annual file at a time) instead of loading everything into pandas. The state subset
is small enough (~0.97 million rows) to profile directly in pandas.


## Run the setup below with the project environment
The first code cell finds the repository root, reloads the helper modules, and runs the offline sanity checks (they use small made-up files and need no raw data).


In [ ]:
import importlib
import sys
from pathlib import Path

# The workspace can be opened at the project folder or one folder above it.
folders = [Path.cwd(), *Path.cwd().parents]
candidates = folders + [folder / "JBG060_ZHL_2026" for folder in folders]
PROJECT_ROOT = None
for folder in candidates:
    if (folder / "data_quality" / "eda_quality_flood" / "flood_eda_data.py").is_file():
        PROJECT_ROOT = folder
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot find JBG060_ZHL_2026 from {Path.cwd()}. Open the project folder first."
    )
if not (PROJECT_ROOT / "raw_data").is_dir():
    raise FileNotFoundError(f"Place the supplied datasets in {PROJECT_ROOT / 'raw_data'}.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

import data_quality.eda_quality_flood.flood_eda_data as f

importlib.reload(f)
from data_quality.eda_quality_flood.check_flood_eda_data import run_checks
from data_quality.eda_quality_flood.flood_eda_data import (
    COUNTRY_LABEL,
    STATE_LABEL,
    area_overview,
    build_profiles,
    expected_file_count,
    flood_files,
)

run_checks()

FLOOD_ROOT = PROJECT_ROOT / "raw_data" / "flood_masks"
BOUNDARIES = PROJECT_ROOT / "raw_data" / "Administrative boundaries" / "ssd_admin1.geojson"
TABLE_DIR = PROJECT_ROOT / "data_quality" / "eda_quality_flood" / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "data_quality" / "eda_quality_flood" / "outputs" / "figures"
for directory in (TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)

total_files = len(flood_files(FLOOD_ROOT))
print(f"Project root: {PROJECT_ROOT}")
print(f"Expected flood-mask files (2 tiles x 2 classes x 26 years): {expected_file_count()}")
print(f"Files present on disk: {total_files}")


## 1. Raw records behind each level
`level_overview` counts how many detection records, unique pixels and unique dates each level covers, plus the date range. These are the raw Parquet records (a record is one location on one three-day composite date); several records can share the same pixel or date.


In [ ]:
overview = area_overview(FLOOD_ROOT, BOUNDARIES)
display(overview)
# Percentage of all detections that fall inside Northern Bahr el Ghazal.
nbw = overview.loc[overview["level"].eq(STATE_LABEL), "rows"].iloc[0]
country_rows = overview.loc[overview["level"].eq(COUNTRY_LABEL), "rows"].iloc[0]
print(f"NBeG detections are {100.0 * nbw / country_rows:.3f}% of the country total.")


## 2. Column-by-column profile
`column_profile` has one row per (level, column) with the number of non-null and null/NaN values, the NaN percentage, the number of distinct values, and where relevant the min, max, mean and standard deviation. Rows for `lat`, `lon` and `cloud_frac` are numeric; `date` gives the date range; `tile` and `flood_type` are small categorical fields.


In [ ]:
profile = build_profiles(FLOOD_ROOT, BOUNDARIES)
display(profile)


In [ ]:
# A wide view focusing on missing values and value ranges per level.
summary = profile.pivot(index="column", columns="level")
nan_view = summary[["nan_count", "nan_pct"]].copy()
nan_view.columns = [f"{level} - {metric}" for metric, level in nan_view.columns]
display(nan_view)

total_nan = int(profile["nan_count"].sum())
print(f"Total null/NaN values across both levels: {total_nan:,}")
if total_nan == 0:
    display(Markdown(
        "**Result: there are no null/NaN values anywhere.** Every record that is present in the "
        "flood-mask files has a value for `date`, `lat`, `lon`, `tile` and `cloud_frac`. "
        "Missing values only appear as *absent rows* (no detection recorded), which this EDA "
        "cannot observe directly."
    ))


## 3. Figures
Two figures summarise the profile: the missing-value counts per column and level, and the spatial coverage (latitude/longitude range) at each level.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained")
# Missing values per column and level.
sns.barplot(data=profile, x="column", y="nan_count", hue="level", ax=axes[0])
axes[0].set_title("Null / NaN counts")
axes[0].set_ylabel("Missing values")
# Spatial range of latitude and longitude.
for column in ("lat", "lon"):
    sub = profile[profile["column"].eq(column)]
    axes[1].barh(
        [f"{level} - {column}" for level in sub["level"]],
        sub["max"].astype(float) - sub["min"].astype(float),
        left=sub["min"].astype(float),
    )
axes[1].set_title("Coordinate ranges")
axes[1].set_xlabel("Degrees")
fig.savefig(FIGURE_DIR / "coverage_ranges.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved figures to {FIGURE_DIR}")


In [ ]:
# Focused missing-value figure (kept separate so the axis scale is clear).
fig, ax = plt.subplots(figsize=(8, 4.2), layout="constrained")
sns.barplot(data=profile, x="column", y="nan_count", hue="level", ax=ax)
ax.set_title("Missing values (all zero in this dataset)")
ax.set_ylabel("NaN / null count")
ax.set_xlabel("")
fig.savefig(FIGURE_DIR / "missing_values.png", dpi=200, bbox_inches="tight")
plt.show()


## 4. Saving the results
The overview and the full column profile are written as CSV tables under `data_quality/eda_quality_flood/outputs/tables/`, and the two figures under `data_quality/eda_quality_flood/outputs/figures/`.


In [ ]:
overview.to_csv(TABLE_DIR / "level_overview.csv", index=False)
profile.to_csv(TABLE_DIR / "column_profile.csv", index=False)
print(f"Saved tables to {TABLE_DIR}")
for path in sorted(TABLE_DIR.iterdir()) + sorted(FIGURE_DIR.iterdir()):
    print(" -", path.name)


## Limitations
- The Parquet files only *contain detections*. There are no rows for places or dates that were not flooded (or not observed), so `0` missing values here does **not** mean complete coverage of South Sudan. Absence of a record is itself a kind of missingness.
- `cloud_frac` is constant at `0.0` in this dataset, so it cannot be used to separate dry land from cloud-obscured land.
- The two tiles cover latitudes 0-10 N, so the dataset is the South Sudan flood-mask product as provided; the northernmost part of the country sits outside these tiles.
